# Week 9: Stylometry 

Stylometry is the quantitative study and statistical analysis of text often for the purpose of authorship attribution.

The inspiration and some of the code in this notebook is drawn from a tutorial posted on  [programminghistorian.org](https://programminghistorian.org/en/lessons/introduction-to-stylometry-with-python). I also used Claude Code to help with the analysis, but I have heavily edited the output from Claude to ensure correctness and to find my own style with the results.  I edited the code to make sure that it only uses Python features that we have discussed in class, and added comments to aid in the readability. 

We are going to look at three different techniques used in field of Stylometry, and discuss some pros and cons. Interestingly, stylometry predates computer programming and in early studies, researchers did all the anlysis by hand.  (I can't imangine counting the lengths of words by hand in an entire play.)

### A. Mendenhall's Word-Length Curves

Thomas Mendenhall proposed that the percentage of 1-letter, 2-letter, 3-letter ... words should form a distinctive curve for each author. He called this the author's "characteristic curve of composition". The work that we are replicating from Mendenhall took place in 1901 and sought to answer the question: did Francis Bacon secretly write the plays of Shakespeare. Without computers, all the words were counted by hand!

### B. Type Token Ratio

Type Token Ratio (or TTR) is the ratio of the number of unique words in a text (Types) to the total number of words (Tokens).  It is used as a measure of linguistic diversity. A higher TTR means that there are more unique words in a text. The value is sensitive to the total number of words, so we typically use a "standardized" type token ratio so that when we make comparisons the we are always comparing the same total number of words.

### C. Burrow's Delta

[Burrow's Delta](https://academic.oup.com/dsh/article/17/3/267/929277) was proposed by John Burrows in 2002, and is the standard baseline in computational authorship attribution. The idea behind Burrow's Delta is to compute how often each word appears in a text, and use the set of most frequent words as fingerprint. The premise is that if we look at multiple texts by the same author, this set of most frequent words shoud be very similar. As we will see below, this technique was used to establish authorship of some of the Federalist papers.

This is also one of the techniques used to detect whether some text was [written by an LLM](https://academic.oup.com/dsh/advance-article/doi/10.1093/llc/fqag064/8714041). 

## A. Mendenhall's Word-Length Curves

One criticism of Mendenhall's work was that word length of poetry might be quite different from prose.  Bacon's writing's were all in prose and Shakespeare's writings were a mixture of prose and poetry.

Mendenhall counted Shakespeare's plays as a whole. We'll do something he couldn't easily do: **separate the verse from the prose**. The Folger Shakespeare Library publishes digital editions where every line of every play is labelled as verse or prose, so we can pull them apart.

| File | What's in it |
|---|---|
| `shakespeare_verse.txt` | All the verse lines from 8 plays: *Hamlet*, *Henry IV* Parts 1 and 2, *Henry V*, *As You Like It*, *Much Ado About Nothing*, *Twelfth Night*, *The Merry Wives of Windsor* |
| `shakespeare_prose.txt` | All the prose lines from the same 8 plays |
| `bacon_prose.txt` | Bacon's *Essays* and *The Advancement of Learning* (two of the works Mendenhall used) |
| `marlowe_plays.txt` | Christopher Marlowe's *Doctor Faustus*, *Tamburlaine* Part 1 and *The Jew of Malta* |

We picked these plays because they have lots of both verse and prose. (Falstaff, for instance, almost always speaks in prose.) Speaker names and stage directions have been removed, leaving only the words spoken on stage. The Shakespeare texts come from the [Folger Shakespeare](https://www.folger.edu/explore/shakespeares-works/) and the Bacon and Marlowe texts from [Project Gutenberg](https://www.gutenberg.org/). (Contact me if you are interested in seeing how these texts were prepared.)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create a dictionary where the keys indicate to whom the text attributed and
# what type of text it is, and the values are a string of the text for analysis
texts = {}
for name in ["shakespeare_verse", "shakespeare_prose", "bacon_prose", "marlowe_plays"]:
    with open(f"mendenhall/texts/{name}.txt", "r") as f:
        texts[name] = f.read()

# A peek at the start of each one to verify that they were read correctly.
for name in texts:
    print(name.upper())
    print(texts[name][:200])
    print()

The next step is to clean up the text a little more, by doing three things:
- replace dashes with spaces
- make everything lower case
- remove puctuation by keeping only the letters

Our goal is to produce a new dictionary that maps the names to a list of cleaned words from each text.

In [ ]:
word_lists = {}
for name in texts:
    current_text = texts[name]

    # Turn dashes into spaces (both the standard hyphen and the unicode dash)
    updated_text = current_text.replace("-", " ")
    updated_text = updated_text.replace("—", " ") # unicode dash

    # Set all words to lower case
    updated_text = updated_text.lower()

    # Get a list of words without any punctuation
    cleaned_list = []

    for token in updated_text.split():
        # build up a word of only letters from the token
        word = ""

        for letter in token:  
            if letter.isalpha(): 
                word = word + letter
        if word != "":
            cleaned_list.append(word)

    word_lists[name] = cleaned_list

for name in word_lists:
    print(f"{name:20} {len(word_lists[name])} words")



One interesting question is how many words do you need to get curves that are reliable?  Mendenhall that 100,000 words were "necessary and sufficient", so our Marlowe sample is on the small side.

### Counting word lengths

The next step is to go though each word list and count the number of words of each length.  Mendenhall plotted words of length 1 to 14 letters, so we will do the same.  This time let's write a function to carry out this task for one word list.  Then we can apply the function to each of our four word lists. To make sure that we are comparing similar things, we will report how many words of each length appear in every 1,000 words.  

In [ ]:
def characteristic_curve(word_list):
    """ return a dictionary of the words of lengths 1 to 14 that appear
        in every 1000 words
    """
    # initialize a new dictionary so that every length has a count of 0
    counts = {}
    for num in range(1, 15):
        counts[num] = 0

    # go through the word list and count each word length
    for word in word_list:
        length = len(word)
        if length < 15:
            counts[length] = counts[length] + 1

    total = len(word_list)
    # normalize each count by divigint by the total and multiplying by 1000
    for length in counts:
        counts[length] = 1000 * counts[length]/total

    return counts

# Comput the curve for each word list
curves = {}
for name in word_lists:
    curves[name] = characteristic_curve(word_lists[name])

# Since we have a dictionary of dictionarys, we can turn it into a dataframe
import pandas as pd
LENGTHS = list(range(1, 15)) # We need a list of numbers from 1 to 14
pd.DataFrame(curves).T.round(1)

In [ ]:
styles = {
    "shakespeare_verse": {"label": "Shakespeare (verse)", "color": "#2a78d6", "linestyle": "-"},
    "shakespeare_prose": {"label": "Shakespeare (prose)", "color": "#2a78d6", "linestyle": "--"},
    "bacon_prose":       {"label": "Bacon (prose)",       "color": "#eb6834", "linestyle": "-"},
    "marlowe_plays":     {"label": "Marlowe (plays)",     "color": "#1baf7a", "linestyle": "-"},
}


def plot_curves(names, title):
    fig, ax = plt.subplots(figsize=(9, 5))
    for name in names:
        s = styles[name]
        # curves[name] is a dictionary, so pull out the values in order of length
        y_values = []
        for length in LENGTHS:
            y_values.append(curves[name][length])
        ax.plot(LENGTHS, y_values, marker="o", linewidth=2,
                color=s["color"], linestyle=s["linestyle"], label=s["label"])
    ax.set_xticks(LENGTHS)
    ax.set_xlabel("Word length (letters)")
    ax.set_ylabel("Words per 1,000")
    ax.set_title(title)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
    plt.show()

In [ ]:
plot_curves(["shakespeare_verse", "shakespeare_prose", "bacon_prose", "marlowe_plays"],
            "Characteristic Curve Comparisons")

## Results

By plotting all four curves we can summarize the conclusion that Mendenhall arrived at as well as the arguments made later by others.

First, we can see that the curve for Bacon and the curves for Shakespeare are quite different which led Mendenhall to conclude that Bacon did not write like Shakespeare. In fact, Mendenhall was surprised to see that the most frequent word length for Shakespeare's verse was 4 letters, since in other texts he had analyzed, the most frequent word lenght was 3 letters.  He was also surprised to see how closely Marlowe's curve matched the curve for Shakespeare's verse. Further analysis by other scholars shows that these curves are better at distinguishing texts from different genres and different eras than they are at distinguishing authors.

## Federalist Papers

We will start by trying Mendehall's curves on the Federalist Papers to see whether it yields any interesting results.

There were 85 Federalist Papers, some with known attribution, and twelve papers where the authorship was unknown or disputed.  We first consturct a dictionary to put the papers in different categories according to what is known about the authorship.

Next, we will read all of the papers, and concatenate the papers in each category together to make one long string.

In [ ]:
papers = {
    'Madison': [10, 14, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48],
    'Hamilton': [1, 6, 7, 8, 9, 11, 12, 13, 15, 16, 17, 21, 22, 23, 24,
                 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 59, 60,
                 61, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77,
                 78, 79, 80, 81, 82, 83, 84, 85],
    'Jay': [2, 3, 4, 5],
    'Shared': [18, 19, 20],
    'Disputed': [49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 62, 63],
    'TestCase': [64]
}

In [ ]:
# A function that compiles all of the text files associated with a single author into a single string
def read_files_into_string(filenames):
    strings = []
    for filename in filenames:
        with open(f'FederalistPapers/federalist_{filename}.txt', 'r') as f:
            strings.append(f.read())
    return '\n'.join(strings)

In [ ]:
# Make a dictionary out of the authors' corpora
federalist_by_author = {}
for author in papers:
    federalist_by_author[author] = read_files_into_string(papers[author])

federalist_by_author["Jay"]

This time we are going to use the nltk (Natural Language Took Kit) to turn our text into tokens. 

In [ ]:
# Load nltk
import nltk
nltk.download('punkt', quiet=True)
%matplotlib inline

# Compare the disputed papers to those written by everyone,
# including the shared ones.
authors = ("Hamilton", "Madison", "Disputed", "Jay", "Shared")

# Transform the authors' corpora into lists of word tokens
federalist_by_author_tokens = {}
federalist_by_author_length_distributions = {}
for author in authors:
    tokens = nltk.word_tokenize(federalist_by_author[author])

    # Filter out punctuation
    # Only add tokens that have at least one alphabetic chacter in them.
    federalist_by_author_tokens[author] = []
    for token in tokens:
        word = ""
        for c in token:
            if c.isalpha():
                word = word + c
        if word != "":
            federalist_by_author_tokens[author].append(word.lower())
        

    # Count how many tokens there are of each length
    token_lengths = []
    for token in federalist_by_author_tokens[author]:
        token_lengths.append(len(token))
    federalist_by_author_length_distributions[author] = nltk.FreqDist(token_lengths)

# For example: how many 3-letter words did Hamilton use?
federalist_by_author_length_distributions["Hamilton"][3]

### Another way to normalize -- use percentages

Mendenhall's idea is that each author has a characteristic *shape* of word-length distribution: some lean on short words, others on longer ones. But a `FreqDist` stores **raw counts**, and our authors wrote very different amounts (Hamilton has about 114,000 tokens, Shared about 5,700). If we plot raw counts, Hamilton's curve towers over everyone else's because he wrote 20 times as much, not because of anything about his style. This length problem will come up again with the type-token ratio.

The solution we used above for the Shakespeare analysis was to look at the frequencies per 1000 words.  In this case, we will turn each count into a **proportion**: what percentage of this author's words are 1 letter long, 2 letters long, and so on.

$$\text{percent of words with length } n = 100 \times \frac{\text{number of words with length } n}{\text{total number of words}}$$

Now every author's curve adds up to 100%, so we compare shapes rather than sizes. The two plots below show the same data, raw on the left and normalized on the right.

(One small trap: NLTK's built-in `FreqDist.plot()` orders the x-axis by how *common* each length is, not by the length itself, which scrambles the curve. So we build the plot ourselves with lengths 1 to 15 in order.)

In [ ]:
import matplotlib.pyplot as plt

lengths = list(range(1, 16))   # word lengths 1 to 15, in order

colors = {"Hamilton": "#2a78d6", "Madison": "#eb6834", "Disputed": "#1baf7a",
          "Jay": "#eda100", "Shared": "#e87ba4"}

fig, (ax_raw, ax_pct) = plt.subplots(1, 2, figsize=(13, 5))

for author in authors:
    dist = federalist_by_author_length_distributions[author]
    total = dist.N()   # total number of tokens for this author

    raw_counts = []
    percentages = []
    for n in lengths:
        raw_counts.append(dist[n])
        percentages.append(100 * dist[n] / total)

    ax_raw.plot(lengths, raw_counts, marker="o", linewidth=2, color=colors[author], label=author)
    ax_pct.plot(lengths, percentages, marker="o", linewidth=2, color=colors[author], label=author)

ax_raw.set_title("Raw counts: dominated by how much each author wrote")
ax_raw.set_ylabel("Number of words")
ax_pct.set_title("Normalized: share of each author's words")
ax_pct.set_ylabel("Percent of words")

for ax in (ax_raw, ax_pct):
    ax.set_xlabel("Word length (letters)")
    ax.set_xticks(lengths)
    ax.legend(title="Author", frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

On the left, it looks as if Hamilton writes completely differently from everyone else, but that's only because he wrote the most. On the right, once we compare percentages, the curves are very similar: they all peak at 2 or 3 letters and then fall off.

Look closely at Hamilton, Madison and Disputed, the three we care about. Their differences are only a percentage point or two, and they don't consistently point one way: at 2 letters the Disputed papers fall between Hamilton and Madison, and at 3 letters they're closer to Hamilton. Jay and Shared stand out more, but they're based on much less text, so some of that is just noise.

So word length on its own is a weak signal for this question. Mendenhall's method is a historical starting point; the next methods look at *which* words authors use, not just how long they are.

## B. Type-Token Ratio

Mendenhall's curves look at how *long* an author's words are. Another of the oldest and simplest measures of style is vocabulary richness or linguistic diversity. Does an author keep reaching for new words, or do they reuse the same ones?

To measure this we need two terms:

* A **token** is every individual word in the text, counted every time it appears.
* A **type** is each *distinct* word, counted only once.

In the sentence *"the cat sat on the mat"* there are **6 tokens** but only **5 types**, because *the* appears twice.

The **type-token ratio (TTR)** is simply

$$\text{TTR} = \frac{\text{number of types}}{\text{number of tokens}}$$

A higher TTR means a more varied vocabulary; a lower TTR means more repetition.

In [ ]:
sentence = "the cat sat on the mat"
tokens = sentence.split()
types = set(tokens)   # a set keeps only one copy of each item

print("Tokens:", tokens, "->", len(tokens))
print("Types: ", types, "->", len(types))
print("TTR:   ", len(types) / len(tokens))

### Getting the words for each author

We'll use NLTK again to split each author's text into tokens. This time we put the steps in a function, `get_words`, so that we can reuse it later.  We use the same steps that we used in the Shakespeare analysis above. 


In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def get_words(text):
    words = []
    for token in nltk.word_tokenize(text):
        # keep the token only if it contains at least one letter
        word = ""
        for c in token:
            if c.isalpha():
                word = word + c
        if word != "":
            words.append(token.lower())
    return words

authors = ["Hamilton", "Madison", "Disputed", "Jay", "Shared"]

federalist_words = {}
for author in authors:
    federalist_words[author] = get_words(federalist_by_author[author])

### Attempt 1: TTR for each author's whole corpus

The obvious thing to do is count tokens and types for everything each author wrote.

In [ ]:
rows = []
for author in authors:
    words = federalist_words[author]
    rows.append({
        "author": author,
        "papers": len(papers[author]),
        "tokens": len(words),
        "types": len(set(words)),
        "TTR": len(set(words)) / len(words),
    })

naive_ttr = pd.DataFrame(rows).set_index("author")
naive_ttr.round(3)

This table shows us how the number of tokens skews the TTR data. Hamilton, who wrote 51 papers, looks like he has the poorest vocabulary of the group, and Jay and the Shared papers look the richest.

Every text uses common words (*the*, *of*, *to*, *and*) over and over. As a text gets longer, the number of tokens keeps growing at the same rate, but you run out of new words, so the number of types grows more and more slowly. TTR therefore goes down as length goes up, whoever the author is.

We can see this by plotting a **vocabulary growth curve**: as we read through each author's words one at a time, how many distinct types have we seen so far?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for author in authors:
    seen = set()
    types_so_far = []
    for word in federalist_words[author]:
        seen.add(word)
        types_so_far.append(len(seen))
    ax.plot(types_so_far, color=colors[author], linewidth=2, label=author)
    # label the end of each line
    ax.annotate(author, (len(types_so_far), types_so_far[-1]),
                xytext=(4, 0), textcoords="offset points", va="center", fontsize=9)

ax.set_xlabel("Tokens read so far")
ax.set_ylabel("Distinct types seen so far")
ax.set_title("Vocabulary growth: new words get rarer the longer you read")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.show()

Every curve bends over: the first 1,000 tokens produce lots of new types, the next 1,000 far fewer. Hamilton's curve just runs much further to the right, so his ratio of types to tokens ends up small.

To compare authors fairly, we need to compare **equal-sized pieces of text**.

### Attempt 2: Standardized TTR

The usual fix is the **standardized type-token ratio (STTR)**:

1. Cut each author's words into chunks of exactly 1,000 tokens (any leftover words at the end are dropped).
2. Compute the TTR of each chunk.
3. Average those TTRs.

Because every chunk is the same length, the length problem disappears.

In [ ]:
CHUNK_SIZE = 1000

def chunk_ttrs(words, size=CHUNK_SIZE):
    ttrs = []
    for start in range(0, len(words) - size + 1, size):
        chunk = words[start:start + size]
        ttrs.append(len(set(chunk)) / size)
    return ttrs

rows = []
for author in authors:
    ttrs = chunk_ttrs(federalist_words[author])
    rows.append({
        "author": author,
        "chunks": len(ttrs),
        "STTR (mean)": sum(ttrs) / len(ttrs),
        "lowest chunk": min(ttrs),
        "highest chunk": max(ttrs),
    })

sttr = pd.DataFrame(rows).set_index("author")
sttr.round(3)

Now the picture looks very different. Hamilton's vocabulary is actually a bit *richer* than Madison's, and the Disputed papers sit right next to Madison.

But an average can hide a lot. Let's look at every 1,000-word chunk individually to see how much the authors overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for i, author in enumerate(authors):
    ttrs = chunk_ttrs(federalist_words[author])
    # spread the dots out a little sideways so they don't sit on top of each other
    xs = []
    for j in range(len(ttrs)):
        xs.append(i + (j % 7 - 3) * 0.04)
    ax.scatter(xs, ttrs, color=colors[author], s=30, alpha=0.7)
    # a black bar marks the average
    mean = sum(ttrs) / len(ttrs)
    ax.plot([i - 0.25, i + 0.25], [mean, mean], color="black", linewidth=2)

ax.set_xticks(range(len(authors)))
labels = []
for author in authors:
    labels.append(f"{author}\n({len(chunk_ttrs(federalist_words[author]))} chunks)")
ax.set_xticklabels(labels)
ax.set_ylabel("TTR of a 1,000-token chunk")
ax.set_title("Each dot is one 1,000-word chunk; black bars are the averages")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.show()

### What does this tell us?

* **Raw TTR is misleading** whenever texts differ in length. 
* **Once length is controlled**, the Disputed papers' vocabulary richness (≈0.384) is very close to Madison's (≈0.386) and lower than Hamilton's (≈0.403). That points toward Madison.
* **But it's weak evidence.** The dots overlap heavily: plenty of Hamilton chunks are less varied than plenty of Madison chunks. Vocabulary richness also depends on *topic* and *genre*, not only on the author. A paper explaining technical constitutional procedure may repeat its key terms more than a rousing opening argument.
* **Jay and Shared** have only 6 and 5 chunks, so we shouldn't read much into their averages.

TTR is a single number boiled down from a whole text, and it is hard to imagine that a single number can capture the nuance we need for analyses like these. 

## C. Burrows's Delta (word-frequency fingerprints)

The idea here is to treat the frequencies of a set of words as an author's **fingerprint**, then measure the **distance** between a known author's fingerprint and the fingerprint of an unknown text. The author with the smallest distance is our best guess.

The best-known version of this is **Burrows's Delta**, published by John Burrows in 2002, and it's still a standard baseline in computational authorship attribution.

**Which words make a good fingerprint?** Surprisingly, not the interesting ones. Words like *liberty*, *militia* or *taxation* depend on what a paper is *about*. The best fingerprints come from the most common words: *the, of, to, which, by, upon*. These **function words** are used constantly and mostly unconsciously, so they reflect habit rather than topic.

The Federalist Papers are the classic case. In 1964, Frederick Mosteller and David Wallace showed that Hamilton and Madison differ sharply in tiny words. The most famous is ***upon***: Hamilton uses it all the time, and Madison almost never does.

### Preparing the texts

This time we keep each paper separate rather than merging them, because we want to attribute the disputed papers **one at a time**.

We also strip the header from each file. The header names the author (e.g. "MADISON"), and we don't want the answer sitting in the data! Everything before the salutation *"To the People of the State of New York"* is header, so we keep only what comes after it. We reuse the `get_words()` function from the type-token section, which removes punctuation and lowercases.

In [ ]:
import re

def read_paper_body(number):
    with open(f'FederalistPapers/federalist_{number}.txt', 'r') as f:
        text = f.read()
    # Split at the salutation and keep everything after it
    # (\W matches the colon or period that follows "New York")
    return re.split(r"To the People of the State of New York\W", text, maxsplit=1)[1]

# words_by_paper[51] will be the list of words in Federalist No. 51
words_by_paper = {}
for group in papers:
    for number in papers[group]:
        words_by_paper[number] = get_words(read_paper_body(number))

candidates = ["Hamilton", "Madison", "Jay"]
len(words_by_paper)

### A warm-up: Mosteller and Wallace's *upon*

Before building the full fingerprint, let's check that a single word really can separate authors. Because the papers differ in length, we use the **rate** (how many times per 1,000 words) rather than raw counts. That's the same normalization idea as before.

In [ ]:
def rate_per_1000(word, words):
    return 1000 * words.count(word) / len(words)

rows = []
for group in ["Hamilton", "Madison", "Jay", "Disputed"]:
    for number in papers[group]:
        rows.append({"group": group, "paper": number,
                     "upon per 1000 words": rate_per_1000("upon", words_by_paper[number])})

# Here is some fancy pandas code for you that summarizes the data
upon = pd.DataFrame(rows)
upon.groupby("group")["upon per 1000 words"].describe().round(2)

Hamilton averages around 3 *upon*s per 1,000 words, and Madison is close to zero. The disputed papers look Madisonian. That's encouraging, but a single word is fragile: one Madison paper with a few *upon*s would throw it off. Burrows's Delta uses **many** words at once.

### Step 1: Choose the words for the fingerprint

We take the 30 most frequent words across all the papers whose authors we know. (Burrows's original method used 30 to 150; 30 is enough here and keeps the tables readable.)

In [ ]:
N_WORDS = 30

# known_words is a complete list of all the words in all the documents
known_words = []
for author in candidates:
    for number in papers[author]:
        known_words.extend(words_by_paper[number])

feature_words = []
for word, count in nltk.FreqDist(known_words).most_common(N_WORDS):
    feature_words.append(word)
print(feature_words)

#### The same step without NLTK

`nltk.FreqDist(known_words).most_common(N_WORDS)` does two jobs in one line: it counts every word, and then it picks out the most common ones. Here is the same step written with only the Python we already know, one job at a time.

1. **Count** every word with a dictionary, just as we have counted things before.
2. **Pick** the most common word, then the most common word we haven't picked yet, and so on, until we have `N_WORDS` of them.

You should get exactly the same list as above.

In [ ]:
# Job 1: count how many times each word appears in the known papers
word_counts = {}
for word in known_words:
    if word in word_counts:
        word_counts[word] = word_counts[word] + 1
    else:
        word_counts[word] = 1

# Job 2: pick the N_WORDS most common words, one at a time
feature_words = []
for i in range(N_WORDS):
    # look through every word for the most common one we haven't picked yet
    best_word = ""
    best_count = 0
    for word in word_counts:
        count = word_counts[word]
        if word not in feature_words and count > best_count:
            best_word = word
            best_count = count
    feature_words.append(best_word)

print(feature_words)

Almost all of these are function words. Only *states* and *government* reflect the topic, which is unavoidable in a collection of essays about government.

### Step 2: Build fingerprints

A fingerprint is just the **relative frequency** of each feature word: what fraction of the text's words is *the*, what fraction is *of*, and so on. Using fractions rather than counts means a long text and a short text can be compared.

We make one fingerprint per candidate author (from all their papers combined) and one for each individual paper.

In [ ]:
def fingerprint(words):
    """ return the relative frequency of each feature word in words """
    counts = nltk.FreqDist(words)
    frequencies = {}
    for word in feature_words:
        frequencies[word] = counts[word] / len(words)
    return pd.Series(frequencies)

def words_from_papers(numbers):
    """ combine the words from several papers into one list """
    all_words = []
    for number in numbers:
        all_words.extend(words_by_paper[number])
    return all_words

# One row per candidate author
fingerprints_by_author = {}
for author in candidates:
    fingerprints_by_author[author] = fingerprint(words_from_papers(papers[author]))
author_fingerprints = pd.DataFrame(fingerprints_by_author).T

# One row per paper (every paper, known and unknown)
fingerprints_by_paper = {}
for number in words_by_paper:
    fingerprints_by_paper[number] = fingerprint(words_by_paper[number])
paper_fingerprints = pd.DataFrame(fingerprints_by_paper).T

# Show the author fingerprints as percentages for readability
(100 * author_fingerprints).round(2)

### Step 3: z-scores put every word on the same scale

**NOTE: I really don't expect anyone to fully understand that math here, but I hope you can follow the discussion enough to get a sense of what it is doing.**

We could compare two fingerprints by subtracting their frequencies word by word. But very common words like *the* and *of* make up roughly 10% and 6% of all words, so even a small *relative* difference in them is a big *absolute* difference. A rarer word like *been* or *government* (about half a percent of the text) could only ever contribute a little, even if one author used it twice as often as the other. A comparison like that would mostly listen to the loudest words. We want every word to get an equal vote.

The strategy is to ask, for each word, **how unusual is this frequency, compared to how much that word normally varies from paper to paper?** That's a **z-score**:

$$z = \frac{\text{frequency in this text} - \text{average frequency}}{\text{standard deviation}}$$

* $z = 0$ means perfectly typical.
* $z = +2$ means two standard deviations *more* often than usual.
* $z = -1$ means one standard deviation *less* often than usual.

We compute the average and standard deviation of each word across all the **known** papers. Now a big jump in a rare word and a big jump in *the* count equally.

In [ ]:
known_papers = []
for author in candidates:
    known_papers.extend(papers[author])

word_means = paper_fingerprints.loc[known_papers].mean()
word_stds = paper_fingerprints.loc[known_papers].std()

def z_scores(fp):
    return (fp - word_means) / word_stds

author_z = z_scores(author_fingerprints)

# How each author differs from the average, word by word
author_z.round(2)

Read across the rows. Each author has a distinctive pattern of pluses and minuses, and *that* pattern is the fingerprint:

* **Madison** uses *on* much more than average (+1.3), as well as *states* and *government*, and uses *to* less (−1.0).
* **Jay** is the most distinctive: lots of *and*, *their* and *they*; far less *the*, *of* and *an*.
* **Hamilton's** z-scores are all close to 0. That's not because he has no style. He wrote 51 of the 69 known papers, so the "average" is mostly Hamilton! This is worth remembering: the baseline depends on which texts you include.

### Step 4: Delta = the average difference in z-scores

Burrows's Delta compares an unknown text with an author one word at a time: take the difference between their z-scores, ignore the sign, and then average those differences over all the words:

$$\Delta = \frac{1}{n} \sum_{i=1}^{n} \left| z_i(\text{unknown}) - z_i(\text{author}) \right|$$

Smaller Delta = more similar. We compute Delta from each disputed paper to each candidate, and also from paper 64, which we know Jay wrote, as a check that the method works.

In [ ]:
def delta(unknown_fp, author):
    return (z_scores(unknown_fp) - author_z.loc[author]).abs().mean()

rows = []
for number in papers["Disputed"] + papers["TestCase"]:
    row = {"paper": number}
    for author in candidates:
        row[author] = delta(paper_fingerprints.loc[number], author)
    rows.append(row)

results = pd.DataFrame(rows).set_index("paper")
results["closest"] = results[candidates].idxmin(axis=1)
results.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ys = range(len(results))

for author in candidates:
    ax.scatter(results[author], ys, color=colors[author], s=60, label=author, zorder=3)

# a grey line joining each paper's Hamilton and Madison scores
for y, (h, m) in enumerate(zip(results["Hamilton"], results["Madison"])):
    ax.plot([h, m], [y, y], color="lightgrey", linewidth=2, zorder=1)

ax.set_yticks(list(ys))
labels = []
for n in results.index:
    if n == 64:
        labels.append(f"No. {n} (Jay, test)")
    else:
        labels.append(f"No. {n}")
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Burrows's Delta (smaller = more similar)")
ax.set_title("Distance from each disputed paper to each candidate author")
ax.legend(title="Author", frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.3)
plt.show()

### What does this tell us?

* **11 of the 12 disputed papers are closest to Madison.** This agrees with Mosteller and Wallace (1964) and with most historians today, who attribute all twelve to Madison.
* **No. 55 comes out closer to Hamilton.** It was also the weakest case in Mosteller and Wallace's study. A single result like this is exactly why we shouldn't treat the method as a verdict: it gives evidence, with some papers much clearer than others.
* **Look at the gaps, not just the winner.** For papers like 53, 57, 58 and 62, Madison is clearly closer. No. 56 is almost a tie, and No. 50 is close.
* **The test case works.** Paper 64, which we know Jay wrote, comes out closest to Jay.

### How much should we trust it? Testing on papers we know

A good habit with any attribution method is to try it on texts where we already know the answer. Below, we take each known Hamilton, Madison and Jay paper, pretend we don't know who wrote it, and see whether Delta picks the right author.

One subtlety: if paper 10 is part of Madison's fingerprint, it's not a fair test to then ask whether paper 10 looks like Madison. So for each test we rebuild the author's fingerprint **without** that paper. This is called **leave-one-out** testing.

In [ ]:
correct = 0
mistakes = []

for true_author in candidates:
    for number in papers[true_author]:
        guesses = {}
        for author in candidates:
            # Build the author's fingerprint, leaving out the paper we're testing
            others = []
            for n in papers[author]:
                if n != number:
                    others.append(n)
            author_fp = fingerprint(words_from_papers(others))
            guesses[author] = (z_scores(paper_fingerprints.loc[number]) - z_scores(author_fp)).abs().mean()
        guess = min(guesses, key=guesses.get)
        if guess == true_author:
            correct += 1
        else:
            mistakes.append((number, true_author, guess))

print(f"Correct: {correct} out of {len(known_papers)} ({100 * correct / len(known_papers):.0f}%)")
print("Mistakes (paper, real author, guess):", mistakes)

Delta gets about 93% of the known papers right (64 of 69). It's good, but not perfect: two Hamilton papers, two Madison papers and one Jay paper are misattributed. So the near-ties among the disputed papers (56, 50, and 55 going to Hamilton) shouldn't be over-read. The strength of the case for Madison comes from **11 of 12 papers pointing the same way**, together with independent evidence like *upon*.

**Things to try:**

* Change `N_WORDS` to 10, 50 or 100 and re-run from Step 1. Do the attributions change?
* Remove *states* and *government* from `feature_words`. Do topic words help or hurt?
* Which words have the biggest z-score differences between Hamilton and Madison? (Try `(author_z.loc["Hamilton"] - author_z.loc["Madison"]).sort_values()`.)

## References

### Mendenhall's word-length curves

* T. C. Mendenhall, ["A Mechanical Solution of a Literary Problem,"](https://en.wikisource.org/wiki/Popular_Science_Monthly/Volume_60/December_1901/A_Mechanical_Solution_of_a_Literary_Problem) *Popular Science Monthly* 60 (December 1901). The source of the story, the quotes and the counting details in this notebook.
* C. B. Williams, ["Mendenhall's Studies of Word-Length Distribution in the Works of Shakespeare and Bacon,"](https://doi.org/10.1093/biomet/62.1.207) *Biometrika* 62, no. 1 (1975): 207–212. The prose-vs-verse objection.
* History of Information, ["Mendenhall and early stylometry"](https://www.historyofinformation.com/detail.php?id=4120).
* *American Scientist*, ["Belles lettres Meets Big Data"](https://www.americanscientist.org/node/3543).
* Folger Shakespeare Library, [Folger Shakespeare downloads](https://www.folger.edu/explore/shakespeares-works/download/). Source of the Shakespeare texts, with verse and prose labelled line by line.
* [Project Gutenberg](https://www.gutenberg.org/). Source of the Bacon and Marlowe texts.

### Burrows's Delta and the Federalist Papers

* John Burrows, ["'Delta': a Measure of Stylistic Difference and a Guide to Likely Authorship,"](https://doi.org/10.1093/llc/17.3.267) *Literary and Linguistic Computing* 17, no. 3 (2002): 267–287. The original paper: z-scores of the most frequent words, averaged into a single distance.
* Frederick Mosteller and David L. Wallace, ["Inference in an Authorship Problem,"](https://doi.org/10.1080/01621459.1963.10500849) *Journal of the American Statistical Association* 58, no. 302 (1963): 275–309. The classic study of the disputed Federalist Papers, including Hamilton's *upon* and Madison's *whilst*. Expanded the next year into the book *Inference and Disputed Authorship: The Federalist* (Addison-Wesley, 1964).
* François Dominic Laramée, ["Introduction to Stylometry with Python,"](https://doi.org/10.46430/phen0078) *Programming Historian* 7 (2018). The tutorial that inspired this notebook, including its Federalist Papers data.
* Shlomo Argamon, ["Interpreting Burrows's Delta: Geometric and Probabilistic Foundations,"](https://doi.org/10.1093/llc/fqn003) *Literary and Linguistic Computing* 23, no. 2 (2008): 131–147. Why Delta works, explained mathematically. For the curious.
* Stefan Evert, Fotis Jannidis, Thomas Proisl, Isabella Reger, Steffen Pielström, Christof Schöch and Thorsten Vitt, ["Understanding and Explaining Delta Measures for Authorship Attribution,"](https://doi.org/10.1093/llc/fqx023) *Digital Scholarship in the Humanities* 32, suppl. 2 (2017): ii4–ii16. Tests many variations of Delta on novels in English, French and German.
* Project Gutenberg, [*The Federalist Papers*](https://www.gutenberg.org/ebooks/1404) (eBook #1404). Source of the Federalist texts.
